# AI Agent 설계 및 구축 실습 과제: 아이디어 독창성 검증 에이전트

- 작성자: 3반 최인서
- 교재 131p 과제 5단계(주제 선정 → 요구사항 분석 → 아키텍처 설계 → Tool 설계 → 프로토타입 구현)를 순서대로 담았다.
- 설계는 교재 120~130p(AI Agent 프로젝트 설계)의 체크리스트와 고려사항을 따랐다.
- 실행 환경: Google Colab, Colab 보안 비밀(우선) 또는 `/content/.env`에 `OPENAI_API_KEY`, `GOOGLE_API_KEY`, `TAVILY_API_KEY`

## Part 1. 주제 선정

서비스 아이디어를 넣으면 유사 서비스를 찾아 **이미 있는 것 / 겹치는 점 / 차별점 후보 / 판정**을 출처와 함께 정리하고, 차별점은 반박 검토를 통과한 것만 남기는 에이전트.

- **해결할 문제**: 과제·공모전·해커톤 아이디어를 낼 때 유사 서비스 조사에 시간이 많이 든다. 이번 과제에서도 첫 주제(기업 분석 에이전트)의 독창성을 손으로 검증하는 데 상당한 시간이 들었고, 결국 사람인·캐치·Deep Research 등과 겹친다는 결론이 나 주제를 바꿨다. 그냥 LLM에 물으면 존재하지 않는 서비스를 지어내거나, 듣기 좋게 "독창적"이라고 답하는 문제도 있다.
- **왜 AI Agent인가**: 아이디어마다 검색어(기능·대상·국내/해외)가 달라 검색 결과를 읽고 다음 검색어를 정하는 판단이 반복된다. 규칙 기반으로는 '유사함'을 판단할 수 없는 비정형 작업이다.
- **AI가 하지 않는 일**: 아이디어를 진행할지는 사람이 정한다. 에이전트는 근거와 판정 초안만 제시한다.

## Part 2. 요구사항 분석 (교재 121~123p)

| 항목 | 질문 | 이 프로젝트의 답 |
| --- | --- | --- |
| 문제 | 무엇을 해결하는가 | 아이디어의 유사 서비스 조사 시간과, LLM의 지어낸 경쟁사·근거 없는 칭찬 |
| 업무 성격 | 정형인가 비정형인가 | 비정형 (아이디어마다 검색 경로와 유사성 판단이 다름) |
| 사용자 | 누가 쓰는가 | 과제·공모전·해커톤을 준비하는 학생, 초기 기획자 (비전문가 포함) |
| 입력 | 무엇을 받는가 | 아이디어 설명 1~3문장, (선택) 대상 사용자, 검색 지역(국내/해외/둘 다) |
| 출력 | 어떤 형태인가 | 판정(높음/보통/낮음) + 유사 서비스 표 + 차별점 후보와 반박 결과 + 확인 불가 항목 |
| 데이터 | 무엇을 참고하는가 | 웹 검색 결과 (서비스 사이트, 기사, 앱스토어, 공모전 수상작 등) |
| 자율성 | 사람 승인이 필요한가 | 리포트 생성은 자동, 진행 여부 판단은 사람 |
| 제약 | 비용·속도·보안 | 검색·LLM 호출 비용, 미공개 아이디어 노출 위험(검색어로 외부 전송됨을 안내) |

**성공 평가 기준** (LLM 결과는 매번 다르므로 미리 정하고 Part 5 평가 셀에서 자동 점검)

1. 유사 서비스를 3개 이상 찾고, 각 서비스에 출처 URL이 있다
2. 리포트의 URL은 모두 실제 검색 결과에 있었다 (지어낸 URL 0개)
3. **정답을 아는 아이디어**(이미 서비스가 많은 아이디어 2개)에서 판정이 '낮음' 또는 '보통'으로 나온다
4. 판정 줄과 확인 불가 섹션이 있다
5. 한 번 실행이 3분 안에 끝난다
6. 반박 에이전트(critique-agent)가 실제로 호출된다 (메인 에이전트가 반박 결과를 지어내지 않았는지 메시지 기록으로 확인)

## Part 3. 아키텍처 설계 (교재 124~126p)

![아키텍처](https://raw.githubusercontent.com/boboinhaco/AI-Agent-Design_Idea-Originality-Agent/main/architecture.png)

```
[입력: 아이디어, 대상 사용자, 검색 지역]
  → [Supervisor] write_todos로 계획
  → [research-agent] 국내외 유사 서비스 검색 (internet_search)
  → [Supervisor] 조사 결과로 초안과 차별점 후보 작성
  → [critique-agent] 차별점마다 같은 기능의 서비스를 다시 검색
  → [Supervisor] 판정 규칙 적용, 최종 리포트 작성
  → [코드] 판정 줄·반박 호출 확인(없으면 재요청) → 출처 URL 대조 → 평가
```

| 결정 | 선택 | 이유 |
| --- | --- | --- |
| 단일 vs 멀티 | 멀티 (Supervisor + Worker 2) | 차별점을 '주장하는 쪽'과 '깨려는 쪽'을 분리해야 확증 편향을 막을 수 있음. 같은 에이전트가 자기 주장을 검토하면 관대해짐 |
| RAG 필요 여부 | 사용 안 함 | 필요한 정보가 실시간 웹 정보라 미리 구축한 Vector DB가 과함 (교재 '간단한 기술' 원칙) |
| 프레임워크 | LangChain 체인 대신 LangGraph 기반 deepagents | 체인(LCEL)은 한 방향 처리에 맞음. 이 과제는 조사→초안→반박→수정으로 단계를 오가며 상태를 유지해야 해서 그래프 구조가 맞음. deepagents는 서브에이전트를 기본 제공해 수업 Multi_agent.ipynb 구조를 그대로 쓸 수 있음 |
| Workflow vs ReAct | 순서는 프롬프트로 정하고 코드로 확인, 서브에이전트 내부는 ReAct | 조사→초안→반박→판정 순서는 지켜져야 하고, 검색어 선택은 자율 판단이 필요함. 반박 단계를 건너뛰는 경우가 있어 실행 함수에서 확인 후 재요청함 |
| HITL 지점 | 1차: 없음 / 확장: 판정 확정 전 사람 확인 | 읽기 전용 도구라 위험이 낮음. 판정은 '초안'으로만 제시 |
| 판정 규칙 | 프롬프트에 기준 명시 | 핵심 기능이 모두 겹치는 서비스가 있으면 낮음, 일부 겹치면 보통, 검색 범위에서 못 찾으면 높음(단, 검색 범위 한계 명시) |

교재 125p 권장대로 간단한 구조(도구 1개, 서브에이전트 2개)로 시작했고, HITL과 비용 측정은 확장안으로 남겼다.

## Part 4. Tool 설계 (교재 127~129p)

도구는 웹 검색 `internet_search` 하나만 두고, 뉴스와 일반 검색은 `topic` 파라미터로 나눈다.

**입력/출력 명세**

| 구분 | 이름 | 타입 | 설명 |
| --- | --- | --- | --- |
| 입력 | `query` (필수) | str | 검색어. 한국어·영어 모두 사용 |
| 입력 | `topic` | `"general"` \| `"news"` | 서비스·제품 페이지는 general, 출시·투자 기사는 news (기본 general) |
| 입력 | `time_range` | `"month"` \| `"year"` \| None | 최근 기간 제한 (기본 없음) |
| 입력 | `max_results` | int | 1~10, 범위를 벗어나면 코드에서 잘라냄 (기본 5) |
| 출력 (성공) | dict | `{"query": ..., "results": [{"title", "url", "content", "score"}, ...]}` | Tavily 검색 결과. 각 `url`은 `SEEN_URLS`에 기록 |
| 출력 (실패) | dict | `{"error": "검색 실패: <사유>"}` | 예외를 던지지 않고 사유를 돌려줘 에이전트가 다른 검색어로 재시도하게 함 |

JSON Schema는 docstring과 타입 힌트에서 자동 생성되며, 5-3 셀 끝에서 실제 생성된 스키마를 출력해 확인한다.

| 고려사항 (교재 128p) | 반영 방법 |
| --- | --- |
| 명확한 이름 | `internet_search`: 이름만으로 웹 검색임을 알 수 있음 |
| 상세한 description | docstring에 용도(유사 서비스·기사 탐색)와 각 인자 설명 작성 → JSON Schema 자동 생성 |
| 최소 권한 원칙 | 직접 추가한 도구는 읽기 전용 검색 하나. deepagents 기본 파일 도구(`write_file` 등)는 에이전트 상태 안의 가상 파일에만 써서 외부에 영향 없음 |
| 오류 처리 | 검색 실패 시 예외 대신 `{"error": "검색 실패: ..."}` 반환 |
| 유사 툴 중복 금지 | 국내·해외·뉴스 검색 툴을 따로 만들지 않고 검색어와 `topic`으로 구분 (수업: 비슷한 툴이 많으면 LLM이 선택을 헷갈림) |
| 출처 검증 | 반환된 URL을 기록해, 리포트의 URL이 실제 검색 결과였는지 코드로 대조 (지어낸 경쟁사 차단) |

## Part 5. 프로토타입 구현

수업 `Multi_agent.ipynb`(create_deep_agent + 서브에이전트 + Tavily) 구조를 바탕으로 출처 검증, 정답 기반 평가, 대화 기억을 더했다.

**수업 내용 활용**: LLM 확률적 특성(평가 기준 사전 정의), 프롬프트 엔지니어링(역할·형식·판정 규칙), Tool Calling·JSON Schema(docstring 기반 도구), Planning(TodoList)·Reflection(반박 에이전트) 패턴, Supervisor-Worker 멀티 에이전트, Streaming, Checkpointer와 thread_id 대화 기억(5-12), 모델 비교 실험(Gemini vs OpenAI)

### 5-1. 환경 설정

In [ ]:
!pip install -q deepagents langchain-openai langchain-google-genai tavily-python python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="/content/.env", override=True)  # .env에서 키 로드

try:
    from google.colab import userdata
    for k in ["OPENAI_API_KEY", "GOOGLE_API_KEY", "TAVILY_API_KEY"]:
        try:
            os.environ[k] = userdata.get(k)  # Colab 보안 비밀이 있으면 우선 적용
        except Exception:
            pass
except ImportError:
    pass

for k in ["OPENAI_API_KEY", "GOOGLE_API_KEY", "TAVILY_API_KEY"]:
    v = os.environ.get(k)
    print(k, "OK" if v else "없음", (v[:5] + "*****") if v else "")  # 키 앞 5자리만 표시

### 5-2. 모델 선택

두 모델을 같은 입력으로 실행해 비교한다.

In [ ]:
MODELS = {
    "openai": "openai:gpt-4o-mini",
    "google": "google_genai:gemini-3.5-flash-lite",
}

### 5-3. 도구: internet_search

In [ ]:
import json
from typing import Literal
from tavily import TavilyClient
from langchain_core.utils.function_calling import convert_to_openai_tool

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
SEEN_URLS: set[str] = set()  # 에이전트가 실제로 받은 검색 결과 URL 모음


def internet_search(
    query: str,
    topic: Literal["general", "news"] = "general",
    time_range: Literal["month", "year"] | None = None,
    max_results: int = 5,
) -> dict:
    """아이디어와 비슷한 서비스, 앱, 기사, 공모전 수상작을 웹에서 검색한다.

    Args:
        query: 검색어 (예: "AI 면접 예상질문 생성 서비스", "AI interview prep company research")
        topic: 출시·투자 기사는 "news", 서비스·제품 페이지는 "general"
        time_range: 최근 기간으로 제한할 때 "month" 또는 "year"
        max_results: 반환할 최대 결과 수 (1~10)
    """
    try:
        res = tavily_client.search(
            query,
            topic=topic,
            time_range=time_range,
            max_results=min(max(max_results, 1), 10),
        )
        for r in res.get("results", []):
            SEEN_URLS.add(r.get("url", "").rstrip("/"))  # 검증용 URL 기록
        return res
    except Exception as e:
        return {"error": f"검색 실패: {e}"}  # 실패 사유를 LLM에 전달


print(json.dumps(convert_to_openai_tool(internet_search), ensure_ascii=False, indent=2))  # docstring으로 생성된 JSON Schema 확인

### 5-4. 서브에이전트 정의

- `research-agent`: 유사 서비스를 넓게 찾는다 (국내 한국어 검색어 + 해외 영어 검색어)
- `critique-agent`: 초안의 차별점 하나하나를 '이미 있는 기능'이라고 가정하고 다시 검색해 반박한다

In [ ]:
research_subagent = {
    "name": "research-agent",
    "description": "아이디어와 비슷한 국내외 서비스·앱·기사를 찾아 기능과 출처를 정리한다.",
    "system_prompt": (
        "너는 시장 조사 리서처야. 아이디어의 핵심 기능을 2~3개 키워드로 나눈 뒤, "
        "한국어 검색어와 영어 검색어를 모두 써서 internet_search로 유사 서비스를 찾아. "
        "서비스마다 이름, 검색 결과에 있던 URL, 핵심 기능, 대상 사용자를 정리해. "
        "검색 결과에 없는 서비스나 URL을 만들어내지 마. 못 찾은 정보는 '확인 불가'로 적어."
    ),
    "tools": [internet_search],
}

critique_subagent = {
    "name": "critique-agent",
    "description": "초안의 차별점마다 이미 그 기능을 가진 서비스가 있는지 다시 검색해 반박한다.",
    "system_prompt": (
        "너는 악마의 변호인 역할의 검토자야. 초안의 차별점 하나하나에 대해 "
        "'이미 이 기능을 가진 서비스가 있다'고 가정하고 internet_search로 찾아. "
        "차별점마다 [이미 있음: 서비스명과 URL] 또는 [차별점 유지: 같은 기능을 찾지 못함]으로 답해. "
        "또 출처 없는 문장과 과장된 표현을 지적해. 칭찬은 하지 마."
    ),
    "tools": [internet_search],
}

### 5-5. 메인(Supervisor) 프롬프트와 판정 규칙

In [ ]:
MAIN_INSTRUCTIONS = """너는 아이디어 독창성 검증가야. 듣기 좋은 말보다 근거를 우선한다.

작업 순서
1. write_todos로 할 일 목록을 먼저 만든다.
2. research-agent에게 유사 서비스 조사를 맡긴다. task 설명에 아이디어 원문, 대상 사용자, 검색 지역을 모두 적는다.
3. 조사 결과만 근거로 초안을 쓰고, 차별점 후보를 2~3개 적는다.
4. critique-agent에게 차별점 반박 검토를 맡긴다. 서브에이전트는 내 파일과 대화를 볼 수 없으므로, task 설명에 아이디어 원문과 차별점 후보 목록을 그대로 붙여 넣는다. [이미 있음]이 나온 차별점은 '이미 있음'으로 고친다. critique-agent 결과를 받기 전에는 최종 리포트를 쓰지 않는다.
   - 차별점마다 critique-agent의 [이미 있음]/[차별점 유지] 결과를 3번 섹션에 그대로 옮기고, 결과와 반대되는 문장을 쓰지 않는다.
5. 아래 규칙으로 판정한다.

판정 규칙
- 낮음: 아이디어의 핵심 기능을 모두 가진 서비스가 1개 이상 있다
- 보통: 핵심 기능 일부가 겹치는 서비스는 있지만, [차별점 유지]가 나온 차별점이 1개 이상 남았다
- 높음: 검색 범위에서 겹치는 서비스를 찾지 못했다 (검색하지 못한 범위를 반드시 함께 적는다)

리포트 형식 (마크다운, 한국어)
판정: 높음|보통|낮음 중 하나 (첫 줄에 이 형식 그대로)
## 1. 아이디어 요약 (핵심 기능 2~3개)
## 2. 유사 서비스 (표: 서비스명 | 출처 URL | 겹치는 기능 | 다른 점)
## 3. 차별점 후보와 반박 결과
## 4. 판정 근거
## 5. 확인 불가 항목과 검색 한계

규칙
- 모든 사실 문장에 출처 URL을 붙인다. URL은 검색 결과에 있던 것만 쓴다.
- 확인되지 않은 내용은 추측하지 말고 5번에 적는다.
"""

### 5-6. 에이전트 생성

설치된 deepagents에는 할 일 목록 미들웨어가 기본 포함되지 않아 `TodoListMiddleware`를 추가한다.

In [ ]:
from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.chat_models import init_chat_model


def build_agent(model_id: str, checkpointer=None):
    model = init_chat_model(model_id, timeout=60, max_retries=2)  # 응답 지연·일시 오류 대비
    return create_deep_agent(
        model=model,
        tools=[internet_search],
        system_prompt=MAIN_INSTRUCTIONS,
        subagents=[research_subagent, critique_subagent],
        middleware=[TodoListMiddleware()],
        checkpointer=checkpointer,  # 대화 기억이 필요할 때만 전달
    )

### 5-7. 출처 검증·판정 추출 함수

LLM은 그럴듯한 서비스와 URL을 지어낼 수 있으므로, 리포트의 URL이 실제 검색 결과(`SEEN_URLS`)에 있었는지 코드로 대조한다.

In [ ]:
import re

URL_RE = re.compile(r"https?://[^\s)\]>|\"']+")
VERDICT_RE = re.compile(r"판정\s*[:：]\s*(높음|보통|낮음)")


def verify_citations(report: str) -> dict:
    cited = {u.rstrip(".,;/") for u in URL_RE.findall(report)}  # 리포트 속 URL 추출
    seen = {u.rstrip("/") for u in SEEN_URLS}
    verified = {u for u in cited if u.rstrip("/") in seen}
    return {
        "cited": len(cited),
        "verified": len(verified),
        "ratio": round(len(verified) / len(cited), 2) if cited else 0.0,
        "unverified_urls": sorted(cited - verified),
    }


def parse_verdict(report: str) -> str | None:
    m = VERDICT_RE.search(report)
    return m.group(1) if m else None  # 판정 줄이 없으면 None


def count_services(report: str) -> int:
    sec = report.split("## 2.")[-1].split("## 3.")[0]  # 유사 서비스 섹션만
    rows = [l for l in sec.splitlines() if l.strip().startswith("|") and "http" in l]
    return len(rows)

### 5-8. 실행 함수

In [ ]:
import time

def to_text(content) -> str:
    if isinstance(content, list):  # Gemini는 content를 블록 리스트로 반환
        return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)
    return content or ""


def called_critique(messages) -> bool:
    for m in messages:
        for tc in getattr(m, "tool_calls", None) or []:
            if (tc.get("args") or {}).get("subagent_type") == "critique-agent":  # 반박 에이전트 호출 여부
                return True
    return False

RETRY_MSG = "critique-agent 검토와 판정까지 끝내야 해. 아직 안 했다면 task 도구로 critique-agent에게 아이디어 원문과 차별점 후보를 넘겨 검토받고, 첫 줄이 '판정: 높음|보통|낮음'인 최종 리포트를 형식대로 다시 써줘."

def run(idea: str, target: str = "", region: str = "국내와 해외", model_key: str = "openai", verbose: bool = True) -> dict:
    SEEN_URLS.clear()  # 실행마다 검증용 URL 초기화
    agent = build_agent(MODELS[model_key])
    question = f"다음 아이디어의 독창성을 검증해줘.\n아이디어: {idea}\n대상 사용자: {target or '미지정'}\n검색 지역: {region}"

    start, final_state, steps = time.time(), None, 0
    for chunk in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": 60},  # 무한 루프 방지
        stream_mode="values",
    ):
        steps += 1
        last = chunk["messages"][-1]
        if verbose:
            print(f"[{type(last).__name__}] {str(getattr(last, 'content', ''))[:100]}")
        final_state = chunk

    report = to_text(final_state["messages"][-1].content)
    for _ in range(2):  # 판정 줄이 없거나 반박 검토를 건너뛰면 최대 2번 재요청
        if parse_verdict(report) and called_critique(final_state["messages"]):
            break
        retry = final_state["messages"] + [{"role": "user", "content": RETRY_MSG}]
        for chunk in agent.stream({"messages": retry}, config={"recursion_limit": 60}, stream_mode="values"):
            steps += 1
            final_state = chunk
        report = to_text(final_state["messages"][-1].content)
    return {
        "idea": idea[:30],
        "model": model_key,
        "seconds": round(time.time() - start, 1),
        "steps": steps,
        "critique_called": called_critique(final_state["messages"]),
        "report": report,
        "verdict": parse_verdict(report),
        "services": count_services(report),
        "citations": verify_citations(report),
    }

### 5-9. 실행: 내 아이디어 검증 (OpenAI)

첫 시연은 이 과제의 첫 주제였던 '기업 분석·면접 준비 에이전트'다. 손으로 조사했을 때 사람인·캐치·Deep Research 등과 겹친다는 결론이 났으므로, 에이전트도 '낮음' 또는 '보통'을 내야 정상이다.

In [ ]:
IDEA = "기업 이름과 지원 직무를 넣으면 최신 뉴스를 조사해 출처가 달린 면접 준비 리포트와 예상 질문을 만들어주는 AI 에이전트"
TARGET = "취업 준비생"

result_openai = run(IDEA, TARGET, model_key="openai")
print("\n=== 최종 리포트 ===\n")
print(result_openai["report"])

### 5-10. 실행: 같은 아이디어 (Google)

같은 입력을 Gemini로 실행해 5-9와 비교한다.

In [ ]:
result_google = run(IDEA, TARGET, model_key="google")
print("\n=== 최종 리포트 ===\n")
print(result_google["report"])

### 5-11. 평가: 성공 기준 자동 점검

**정답 기반 테스트**: 이미 서비스가 많은 아이디어를 넣었을 때 '높음'이 나오면 에이전트가 틀린 것이다. 두 번째 사례로 음식 사진 칼로리 앱을 추가해 점검한다.

In [ ]:
KNOWN_CASES = [
    {"idea": IDEA, "target": TARGET, "expect": {"낮음", "보통"}},
    {"idea": "음식 사진을 찍으면 AI가 칼로리와 영양 성분을 알려주는 앱", "target": "다이어트하는 사람", "expect": {"낮음", "보통"}},
]

case2 = run(KNOWN_CASES[1]["idea"], KNOWN_CASES[1]["target"], model_key="openai", verbose=False)
print(case2["report"][:1500])

In [ ]:
import pandas as pd


def summarize(r: dict, expect: set) -> dict:
    c = r["citations"]
    return {
        "아이디어": r["idea"],
        "모델": r["model"],
        "판정": r["verdict"],
        "반박 에이전트 호출": r["critique_called"],
        "정답 기준 통과": r["verdict"] in expect,
        "유사 서비스 수(3개 이상)": r["services"],
        "인용 URL 수": c["cited"],
        "검증된 URL 비율": c["ratio"],
        "지어낸 URL 의심 수": len(c["unverified_urls"]),
        "확인 불가 섹션": "확인 불가" in r["report"],
        "실행 시간(초)": r["seconds"],
        "3분 이내": r["seconds"] <= 180,
    }


rows = [
    summarize(result_openai, KNOWN_CASES[0]["expect"]),
    summarize(result_google, KNOWN_CASES[0]["expect"]),
    summarize(case2, KNOWN_CASES[1]["expect"]),
]
pd.DataFrame(rows)

In [ ]:
for r in [result_openai, result_google, case2]:
    print(r["model"], r["idea"], "검증 안 된 URL:", r["citations"]["unverified_urls"])  # 직접 열어 확인할 목록

### 5-12. 대화 기억으로 후속 질문 (checkpointer + thread_id)

수업에서 배운 MemorySaver와 thread_id로, 검증 결과를 바탕으로 같은 대화에서 후속 요청을 이어간다.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

chat_agent = build_agent(MODELS["openai"], checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "idea-1"}, "recursion_limit": 60}  # 같은 thread_id면 대화가 이어짐

first = chat_agent.invoke(
    {"messages": [{"role": "user", "content": f"다음 아이디어의 독창성을 검증해줘.\n아이디어: {IDEA}"}]},
    config=config,
)
for _ in range(2):  # 판정 줄이나 반박 검토가 빠지면 같은 thread에서 재요청
    if parse_verdict(to_text(first["messages"][-1].content)) and called_critique(first["messages"]):
        break
    first = chat_agent.invoke({"messages": [{"role": "user", "content": RETRY_MSG}]}, config=config)

print(to_text(first["messages"][-1].content)[:1500])

In [ ]:
follow = chat_agent.invoke(
    {"messages": [{"role": "user", "content": "방금 리포트 3번에서 [차별점 유지]가 나온 차별점을 먼저 그대로 적고, 그 차별점을 더 키우려면 기능을 어떻게 바꾸면 좋을지 2가지만 제안해줘. [차별점 유지]가 하나도 없었다면 그렇다고 밝히고 새 차별점 방향을 제안해줘."}]},
    config=config,
)
print(to_text(follow["messages"][-1].content))  # 이전 검증 결과를 기억하고 답하는지 확인

## Part 6. 결과 정리와 한계

**실행 결과** (5-11 평가 표, 2026-10-13 실행 기준)

| 항목 | 면접 리포트 (gpt-4o-mini) | 면접 리포트 (Gemini) | 칼로리 앱 (gpt-4o-mini) |
| --- | --- | --- | --- |
| 판정 / 정답 기준 통과 | 낮음 / 통과 | 낮음 / 통과 | 낮음 / 통과 |
| 반박 에이전트 호출 | O | O | O |
| 유사 서비스 수 (3개 이상) | 10 | 3 | 5 |
| 검증된 URL 비율 | 1.00 | 0.75 (1개 미검증: perplexity.ai) | 1.00 |
| 확인 불가 섹션 | 있음 | 있음 | 있음 |
| 실행 시간 (초, 3분 이내) | 44.9 | 26.3 | 35.1 |

- 성공 기준 6개 중 '지어낸 URL 0개'만 Gemini에서 미달했다. 미검증 URL은 실제 존재하는 사이트지만 이번 검색 결과에는 없었고, 모델이 사전 지식으로 붙인 출처라 검증 계층이 의도대로 잡아냈다.
- LLM 결과는 실행마다 달라 위 숫자도 다시 실행하면 바뀐다. 그래서 기준을 미리 정하고 매번 같은 셀로 점검하도록 만들었다.
- 모델 비교: Gemini는 리포트 문장이 길고 근거 설명이 자세했지만 유사 서비스 수가 적었고, gpt-4o-mini는 더 많은 서비스를 찾았지만 작업 순서를 덜 지켰다(아래 개선 과정 참고).

**실험 중 발견한 문제와 개선 과정**

| 문제 | 원인 | 해결 |
| --- | --- | --- |
| 검색이 전부 실패하고 리포트에 지어낸 URL이 들어감 | Tavily 키 자리에 다른 키가 들어가 인증 실패, 에이전트가 검색 없이 리포트 작성 | 키를 Colab 보안 비밀에서 우선 읽도록 수정. 검증 계층(URL 대조)이 없었다면 그럴듯한 리포트로 보였을 사례 |
| Gemini 실행에서 TypeError | Gemini는 응답 content를 문자열이 아닌 블록 리스트로 반환 | `to_text()`로 문자열로 변환 후 판정 추출 |
| critique-agent가 "초안을 주세요"라고만 답함 | 서브에이전트는 메인의 파일·대화를 볼 수 없는데 초안을 넘기지 않음 | 프롬프트에 "task 설명에 아이디어 원문과 차별점 후보를 붙여 넣는다" 추가 |
| gpt-4o-mini가 반박 단계를 건너뛰고 결과를 스스로 씀, 판정 줄 누락 | 작은 모델이 여러 단계 지시를 끝까지 따르지 못함 | 프롬프트만으로는 해결되지 않아, 실행 함수에서 반박 호출(`called_critique`)과 판정 줄을 확인하고 없으면 최대 2번 재요청 |
| "[반박 성공] → 서비스 없음"처럼 결과를 반대로 씀 | '반박 성공/실패' 라벨이 직관적이지 않음 | 라벨을 [이미 있음] / [차별점 유지]로 변경 |

**이 에이전트 자체의 독창성 검증**: 설계 단계에서 직접 검색해 보니 아이디어 검증 도구도 이미 있다. [Preuve AI](https://preuve.ai/blog/best-startup-validation-tools-2026), [IdeaProof](https://ideaproof.io/versus/ideaproof-vs-chatgpt), [ValidMyIdea](https://alternativeto.net/software/validmyidea/about/) 같은 해외 서비스가 경쟁사 탐색과 시장 분석을 해 주고, ChatGPT·Gemini·Perplexity의 Deep Research도 비슷한 조사를 할 수 있다. 그래서 이 프로젝트의 판정을 스스로 적용하면 '보통'에 가깝다. 남는 차별점은 다음과 같다.

- 해외 도구는 창업 아이디어의 시장 규모·수익성 점수에 집중하지만, 이 에이전트는 학생 과제·공모전 아이디어의 '이미 있는가'에만 집중하고 한국어·국내 서비스를 함께 찾는다
- 차별점을 깨는 것만 맡은 반박 에이전트를 둬서 듣기 좋은 판정을 막는다
- 리포트의 URL을 실제 검색 결과와 코드로 대조하고, 정답을 아는 아이디어로 판정을 테스트해 결과를 숫자로 보여준다

**한계와 개선 계획**
- 작은 모델은 다단계 지시를 건너뛸 수 있다 → 코드 확인·재요청으로 보완했지만 재요청만큼 시간과 비용이 늘어난다. 확장안으로 반박 단계를 LangGraph 노드로 고정하는 Workflow 구조를 검토
- 웹 검색에 잡히지 않는 서비스(비공개, 앱 내부 기능)는 못 찾는다 → 판정에 검색 한계를 함께 표시
- '유사하다'는 판단은 LLM 주관이 들어간다 → 정답 사례를 늘려 판정 정확도를 측정
- 미공개 아이디어가 검색어로 외부 서비스에 전송된다 → 사용 전 안내, 확장안으로 키워드만 추출해 검색
- 구조는 deepagents 공식 deep_research 예제와 비슷하다 → 차별은 반박 구조·판정 규칙·검증 계층에 있음
- 판정 확정 전 사람 확인(`interrupt_on`)과 토큰 비용 측정은 확장안